# Assignment 1 — Google Colab CPU runner

This notebook clones the project into Colab, uses Colab's preinstalled scientific stack, runs Assignment 1 on CPU, and packages the generated artifacts for download.

## Before running

> If you previously ran `pip install -r requirements.txt` on this Colab server, remove/restart that server and reconnect before continuing. That command replaces NumPy, SciPy, scikit-learn, PyTorch, and other packages while the old versions may still be loaded in memory.

Run the cells from top to bottom on a fresh CPU runtime.

In [1]:
import os
import platform
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
Working directory: /content


## Get the project

The cell clones the repository the first time and pulls new commits on later runs.

In [2]:
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/MinhTrinhh/deep-learning-landing-page.git"
REPOSITORY_DIR = Path("/content/deep-learning-landing-page")

if (REPOSITORY_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

print("Repository ready at:", REPOSITORY_DIR)

Repository ready at: /content/deep-learning-landing-page


## Install only missing Colab extras

Colab already provides a mutually compatible PyTorch/NumPy/SciPy/scikit-learn stack. The repository's fully pinned `requirements.txt` is intended for a fresh local virtual environment, so this notebook deliberately does not install it. Only lightweight packages missing from Colab are added, without replacing their dependencies.

In [3]:
from importlib.util import find_spec

colab_extras = {
    "thop": "thop==0.1.1.post2209072238",
    "umap": "umap-learn",
}
missing_packages = [
    package_name
    for import_name, package_name in colab_extras.items()
    if find_spec(import_name) is None
]

if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", *missing_packages],
        check=True,
    )
else:
    print("All Colab extras are already installed.")

All Colab extras are already installed.


In [4]:
from importlib.metadata import version

import matplotlib
import numpy as np
import scipy
import seaborn
import sklearn
import thop
import torch
import torchvision
import umap

versions = {
    "PyTorch": torch.__version__,
    "Torchvision": torchvision.__version__,
    "NumPy": np.__version__,
    "SciPy": scipy.__version__,
    "Scikit-learn": sklearn.__version__,
    "Matplotlib": matplotlib.__version__,
    "Seaborn": seaborn.__version__,
    "UMAP-learn": version("umap-learn"),
    "THOP": version("thop"),
}
for package, package_version in versions.items():
    print(f"{package}: {package_version}")
print("Execution device: CPU")

PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
NumPy: 2.1.3
SciPy: 1.16.3
Scikit-learn: 1.6.1
Matplotlib: 3.10.0
Seaborn: 0.13.2
UMAP-learn: 0.5.12
THOP: 0.1.1-2209072238
Execution device: CPU


## Configure and run an experiment

Choose one model family and whether to use augmentation or regenerate the EDA outputs. The project code currently trains on CPU.

In [5]:
MODEL = "linear"             # "linear", "mlp", or "all"
USE_AUGMENTATION = True       # False adds --no-aug
RUN_EDA = False               # True regenerates EDA before training

assert MODEL in {"linear", "mlp", "all"}
print({"model": MODEL, "augmentation": USE_AUGMENTATION, "eda": RUN_EDA})

{'model': 'linear', 'augmentation': True, 'eda': False}


In [6]:
CODE_DIR = REPOSITORY_DIR / "Assignment1" / "code"
command = [sys.executable, "-u", "main.py", "--model", MODEL]
command.append("--aug" if USE_AUGMENTATION else "--no-aug")
if RUN_EDA:
    command.append("--eda")

experiment_environment = os.environ.copy()
experiment_environment["PYTHONUNBUFFERED"] = "1"

print("Running:", " ".join(command), flush=True)
with subprocess.Popen(
    command,
    cwd=CODE_DIR,
    env=experiment_environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    assert process.stdout is not None
    for output_line in process.stdout:
        print(output_line, end="", flush=True)
    return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)

Running: /usr/bin/python3 -u main.py --model linear --aug
Epoch 1/10000:
Mean train loss: 1.1048543631183028
Train Accuracy: 0.6033888888888889
Mean validate loss:  0.8088128243883451
Validate Accuracy: 0.729
Saved new best checkpoint: /content/deep-learning-landing-page/Assignment1/checkpoints/best_linear_aug.pt
Epoch 2/10000:
Mean train loss: 0.9809312806310246
Train Accuracy: 0.648425925925926
Mean validate loss:  0.767603300511837
Validate Accuracy: 0.7343333333333333
Saved new best checkpoint: /content/deep-learning-landing-page/Assignment1/checkpoints/best_linear_aug.pt
Epoch 3/10000:
Mean train loss: 0.9656477886918596
Train Accuracy: 0.6564814814814814
Mean validate loss:  0.7533759847283363
Validate Accuracy: 0.7353333333333333
Saved new best checkpoint: /content/deep-learning-landing-page/Assignment1/checkpoints/best_linear_aug.pt
Epoch 4/10000:
Mean train loss: 0.9603459349175765
Train Accuracy: 0.6594444444444445
Mean validate loss:  0.7649599015712738
Validate Accuracy: 0.

KeyboardInterrupt: 

## Inspect and download artifacts

The final cell creates one ZIP archive containing `checkpoints/` and `outputs/`. Download it from the Colab Contents panel in VS Code.

In [ ]:
ASSIGNMENT_DIR = REPOSITORY_DIR / "Assignment1"
artifact_roots = [ASSIGNMENT_DIR / "checkpoints", ASSIGNMENT_DIR / "outputs"]

for artifact_root in artifact_roots:
    print(f"\n{artifact_root.relative_to(REPOSITORY_DIR)}/")
    if artifact_root.exists():
        for artifact in sorted(path for path in artifact_root.rglob("*") if path.is_file()):
            print(" -", artifact.relative_to(artifact_root))
    else:
        print(" - No artifacts generated yet")

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

archive_path = Path("/content/assignment1-artifacts.zip")
included_files = 0

with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for directory_name in ("checkpoints", "outputs"):
        directory = ASSIGNMENT_DIR / directory_name
        if not directory.exists():
            print(f"Skipped missing directory: {directory}")
            continue

        for source_path in sorted(path for path in directory.rglob("*") if path.is_file()):
            archive_name = Path("Assignment1") / source_path.relative_to(ASSIGNMENT_DIR)
            archive.write(source_path, archive_name)
            included_files += 1

if included_files == 0:
    archive_path.unlink(missing_ok=True)
    raise FileNotFoundError("No checkpoint or output files were found. Run an experiment first.")

archive_size_mb = archive_path.stat().st_size / (1024 ** 2)
print(f"Created {archive_path} with {included_files} files ({archive_size_mb:.2f} MB)")
print("Download it from the VS Code Colab Contents view.")